In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PostgresConnector") \
    .config("spark.jars","/usr/lib/spark/jars/postgresql-42.7.4.jar")\
    .getOrCreate()


In [ ]:
# Question 43: Monthly Active Users (MAU) - Count of employees active in each month.
# Concept: Expanding date ranges into monthly records
employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})

department_employee_df = spark.read.jdbc(url="jdbc:postgresql://localhost:5432/employees",table="employees.department_employee",
                         properties={"user": "vagrant", "password": "vagrant","driver": "org.postgresql.Driver"})
# Solution:
# Generate Month Series (similar to Q41)
min_date = department_employee_df.agg(min("from_date")).collect()[0][0].replace(day=1)
max_date = department_employee_df.agg(max("to_date")).collect()[0][0].replace(day=1)
date_list = pd.date_range(min_date, max_date, freq='MS')
months_df = spark.createDataFrame(date_list.to_frame(index=False), DateType()).toDF("month_start")

# Range Join: Active if month_start is between from_date and to_date
active_counts = months_df.crossJoin(department_employee_df) \
    .filter((col("month_start") >= trunc(col("from_date"), "month")) & 
            (col("month_start") <= trunc(col("to_date"), "month"))) \
    .groupBy("month_start") \
    .agg(countDistinct("employee_id").alias("active_employees"))

active_counts.show()
